#  Complete Data Validation & Recovery Notebook

This notebook performs **comprehensive data validation and recovery**:

1. **Checks all samples** for completeness
2. **Identifies missing**: images, responses, evaluations
3. **Recovers missing data**:
   - Fetches missing images from Cauldron
   - Runs VLM inference for missing responses (with full GPU metrics)
   - Runs Glider/VLM Judge for missing evaluations
4. **Handles errors**: Marks failed responses with `ok=False`

## Safety
- `dry_run = True` by default
- Set `config.dry_run = False` to enable recovery

In [20]:
# === Cell 1: Setup & Imports ===
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
ARTEMIS_DIR = NOTEBOOK_DIR.parent.parent
ROOT_DIR = ARTEMIS_DIR.parent

for p in [str(ARTEMIS_DIR), str(ROOT_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)


In [21]:

import os
import io
import json
import logging
import threading
from datetime import datetime
from dataclasses import dataclass, field
from typing import Dict, List, Any, Optional, Set, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict

import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from sqlalchemy import text
from PIL import Image
from IPython.display import display, HTML, Markdown

# Suppress HTTP noise
for logger_name in ['httpx', 'openai', 'httpcore', 'urllib3']:
    logging.getLogger(logger_name).setLevel(logging.WARNING)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(name)-12s | %(levelname)-7s | %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger('VALIDATION')

print(f" ARTEMIS_DIR: {ARTEMIS_DIR}")
print(" Imports done!")

📁 ARTEMIS_DIR: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/artemis_final
✅ Imports done!


In [22]:
# === Cell 2: Configuration ===
@dataclass
class ValidationConfig:
    """Validation and recovery settings."""
    # Safety
    dry_run: bool = False
    
    # Limits
    max_recovery_samples: int = 10000
    batch_size: int = 25
    
    # Expected data
    expected_models: int = 5
    
    # Inference settings
    models_yaml: str = str(ARTEMIS_DIR / 'ares' / 'configs' / 'models.yaml')
    temperature: float = 0.0
    max_tokens: int = 512
    train_ratio: float = 0.70
    val_ratio: float = 0.15
    run_id: str = field(default_factory=lambda: f"recovery_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
    
    # GPU endpoints
    gpu_endpoints: Dict[str, str] = field(default_factory=lambda: {
        'deepseek_ocr': 'http://localhost:9003/metrics',
        'qwen2_5_vl_3b': 'http://localhost:9001/metrics',
        'qwen2_5_vl_7b': 'http://localhost:9001/metrics',
        'qwen3_vl_8b_thinking': 'http://localhost:9000/metrics',
        'gemma_3_27b': 'http://localhost:9000/metrics',
    })
    
    # Evaluation endpoints
    glider_ports: List[int] = field(default_factory=lambda: [8806])
    vlm_judge_ports: List[int] = field(default_factory=lambda: [8807])

config = ValidationConfig()

print("️ Configuration:")
print(f"    Dry Run: {config.dry_run} {'(will NOT modify data)' if config.dry_run else '️ WILL MODIFY DATA'}")
print(f"    Max recovery samples: {config.max_recovery_samples}")

⚙️ Configuration:
   🔒 Dry Run: False ⚠️ WILL MODIFY DATA
   📊 Max recovery samples: 10000


In [23]:
# === Cell 3: Database Connection ===
from ares.db.connection import get_engine, test_connection
from ares.db.operations import insert_responses, insert_images, insert_samples, insert_evaluations
from ares.configs.db_config import MODEL_NAMES, MODEL_PREFIXES

print(" Testing database connection...")
if not test_connection():
    raise RuntimeError("Database connection failed!")

engine = get_engine()
logger.info(f"Database connected! Models: {MODEL_NAMES}")

16:45:11 | VALIDATION   | INFO    | Database connected! Models: ['deepseek_ocr', 'qwen2_5_vl_3b', 'qwen2_5_vl_7b', 'qwen3_vl_8b_thinking', 'gemma_3_27b']


🔗 Testing database connection...
✓ Connected to database. Server time: 2025-12-07 21:43:55.995188+00:00


---
#  Section 1: Comprehensive Validation

Check all samples for missing images, responses, and evaluations.

In [24]:
# === Cell 4: Table Overview ===
tables = ['vlm_samples', 'vlm_images', 'vlm_responses', 'vlm_evaluations']
row_counts = {}

with engine.connect() as conn:
    for table in tables:
        result = conn.execute(text(f"SELECT COUNT(*) FROM {table}"))
        row_counts[table] = result.fetchone()[0]

print(" Table Row Counts:")
print("=" * 50)
for table, count in row_counts.items():
    print(f"{table:25} {count:>15,} rows")
print("=" * 50)

📊 Table Row Counts:
vlm_samples                        55,890 rows
vlm_images                         45,674 rows
vlm_responses                     279,450 rows
vlm_evaluations                   269,464 rows


In [25]:
# === Cell 5: Find All Issues ===
# Query to find samples with their completeness status

query_sample_status = """
WITH sample_status AS (
    SELECT 
        s.sample_id,
        s.source_config,
        s.source_index,
        s.data_split,
        s.prompt_text,
        s.ground_truth,
        s.image_id,
        -- Check if image exists
        CASE WHEN i.image_id IS NOT NULL THEN true ELSE false END as has_image,
        -- Count responses (ok=true only)
        (SELECT COUNT(DISTINCT model_name) FROM vlm_responses r WHERE r.sample_id = s.sample_id AND r.ok = true) as response_count,
        -- Get models with responses
        (SELECT ARRAY_AGG(DISTINCT model_name) FROM vlm_responses r WHERE r.sample_id = s.sample_id AND r.ok = true) as models_with_responses,
        -- Count evaluations
        (SELECT COUNT(DISTINCT model_name) FROM vlm_evaluations e WHERE e.sample_id = s.sample_id) as eval_count
    FROM vlm_samples s
    LEFT JOIN vlm_images i ON s.image_id = i.image_id
)
SELECT * FROM sample_status
WHERE has_image = false OR response_count < :expected_models OR eval_count < response_count
ORDER BY source_config, sample_id
"""

with engine.connect() as conn:
    issues_df = pd.read_sql(text(query_sample_status), conn, params={'expected_models': config.expected_models})

# Categorize issues
missing_images = issues_df[~issues_df['has_image']]
missing_responses = issues_df[issues_df['response_count'] < config.expected_models]
missing_evals = issues_df[issues_df['eval_count'] < issues_df['response_count']]

print(f"\n VALIDATION SUMMARY")
print("=" * 60)
print(f"   Total samples in DB:         {row_counts['vlm_samples']:>10,}")
print(f"    Missing images:            {len(missing_images):>10,}")
print(f"    Incomplete responses:      {len(missing_responses):>10,}")
print(f"    Missing evaluations:       {len(missing_evals):>10,}")
print("=" * 60)

if len(issues_df) > 0:
    print("\n Issues by source_config (top 20):")
    display(issues_df.groupby('source_config').size().reset_index(name='count').head(20))
else:
    print("\n All data is complete!")


📋 VALIDATION SUMMARY
   Total samples in DB:             55,890
   ❌ Missing images:                     0
   ❌ Incomplete responses:           8,827
   ❌ Missing evaluations:              500

📊 Issues by source_config (top 20):


,source_config,count
0,ai2d,260
1,aokvqa,310
2,chart2text,310
3,chartqa,270
4,clevr,259
5,cocoqa,260
6,datikz,1010
7,diagram_image_to_text,25
8,docvqa,1009
9,dvqa,1012


---
#  Section 2: Initialize Recovery Clients

Setup VLM clients, GPU metrics, and evaluation pipeline.

In [26]:
# === Cell 6: Initialize All Clients ===

vlm_client = None
gpu_client = None
scorer = None
model_specs = None
eval_pipeline = None

if config.dry_run:
    print("⏭️ Dry run mode - skipping client initialization")
else:
    print(" Initializing recovery clients...")
    
    # VLM Client for inference
    try:
        from inference_engine.client import WhichVLMClient
        from ares.evaluation.evaluation import Scorer
        from ares.metrics.metrics_client import GPUMetricsClient
        from ares.utils.common_utils import return_model_specs
        from ares.data.dataset_loader import CauldronLoader
        from ares.evaluation.sample_processor import image_to_png_bytes, compute_image_hash
        
        vlm_client = WhichVLMClient.from_yaml(config.models_yaml)
        gpu_client = GPUMetricsClient(endpoints=config.gpu_endpoints)
        scorer = Scorer()
        model_specs = return_model_specs()
        print(f"    VLM client initialized")
    except Exception as e:
        logger.error(f"Failed to initialize VLM client: {e}")
    
    # Evaluation pipeline
    try:
        from inference_engine.runners import OpenAIStyleRunner
        from inference_engine.config import ModelEndpoint
        from ares.evaluation.router_eval_pipeline import RouterEvalPipeline
        
        endpoints = []
        for i, port in enumerate(config.glider_ports):
            endpoints.append(ModelEndpoint(
                name=f"glider-{i+1}", model_id="PatronusAI/glider",
                base_url=f"http://localhost:{port}/v1", api_key="EMPTY",
                pricing={}, extra_params={}
            ))
        for i, port in enumerate(config.vlm_judge_ports):
            endpoints.append(ModelEndpoint(
                name=f"vlm-judge-{i+1}", model_id="nvidia/Llama-4-Scout-17B-16E-Instruct-FP8",
                base_url=f"http://localhost:{port}/v1", api_key="EMPTY",
                pricing={}, extra_params={}
            ))
        
        runner = OpenAIStyleRunner(models=endpoints, request_timeout_s=180, max_workers=64)
        eval_pipeline = RouterEvalPipeline(
            engine=engine, runner=runner,
            glider_model_names=[f"glider-{i+1}" for i in range(len(config.glider_ports))],
            vlm_judge_model_names=[f"vlm-judge-{i+1}" for i in range(len(config.vlm_judge_ports))],
            tracker_path="recovery_eval_progress.json",
            use_glider=True, use_vlm_judge=True,
        )
        print(f"    Evaluation pipeline initialized")
    except Exception as e:
        logger.error(f"Failed to initialize eval pipeline: {e}")
        eval_pipeline = None
    
    print(" All clients ready!")

🔧 Initializing recovery clients...
Configured models:
  id=0 name=deepseek_ocr prefix=deepseek_ocr__
  id=1 name=qwen2_5_vl_3b prefix=qwen2_5_vl_3b__
  id=2 name=qwen2_5_vl_7b prefix=qwen2_5_vl_7b__
  id=3 name=qwen3_vl_8b_thinking prefix=qwen3_vl_8b_thinking__
  id=4 name=gemma_3_27b prefix=gemma_3_27b__
   ✅ VLM client initialized
   ✅ Evaluation pipeline initialized
✅ All clients ready!


---
#  Section 3: Recovery Functions

In [27]:
# === Cell 7: Recovery Helper Functions ===
# FULLY CORRECTED VERSION with all API fixes

recovery_logger = logging.getLogger('RECOVERY')

def fetch_image_from_cauldron(source_config: str, sample_idx: int, sample_id: str, image_id: str) -> Optional[bytes]:
    """Fetch image from Cauldron and store in DB."""
    try:
        samples = CauldronLoader.load_samples(source_config, n_samples=sample_idx + 10, random_sample=False)
        if sample_idx >= len(samples):
            recovery_logger.warning(f"Sample index {sample_idx} out of range for {source_config}")
            return None
            
        qa = CauldronLoader.extract_qa(samples[sample_idx], source_config)
        if qa and qa.get('image'):
            image = qa['image']
            image_bytes = image_to_png_bytes(image)
            image_hash = compute_image_hash(image_bytes)
            
            image_record = {
                'image_id': image_id,
                'image_bytes': image_bytes,
                'image_hash': image_hash,
                'img_width': image.width,
                'img_height': image.height,
                'img_aspect_ratio': image.width / image.height if image.height > 0 else 1.0,
                'img_file_size_bytes': len(image_bytes),
                'image_path': None, 'image_cache_root': None,
                'cauldron_image_asset': None,
                'cauldron_lookup_key': f"{source_config}_{sample_idx}",
            }
            insert_images([image_record])
            recovery_logger.info(f"   Stored image for {sample_id}")
            return image_bytes
    except Exception as e:
        recovery_logger.error(f"Failed to fetch image for {sample_id}: {e}")
    return None


def run_inference_for_models(
    sample_id: str, prompt: str, ground_truth: str, image_bytes: bytes,
    models_to_run: List[str], source_config: str
) -> List[Dict]:
    """Run VLM inference with full GPU metrics. Mark errors properly."""
    # Import confidence function
    from ares.evaluation.confidence import estimate_confidence
    
    response_records = []
    image = Image.open(io.BytesIO(image_bytes))
    
    for model_name in models_to_run:
        model_prefix = MODEL_PREFIXES[MODEL_NAMES.index(model_name)] if model_name in MODEL_NAMES else model_name
        model_id = MODEL_NAMES.index(model_name) if model_name in MODEL_NAMES else 0
        
        try:
            # FIX 1: Get GPU metrics using get_gpu_summary()
            gpu_metrics = gpu_client.get_gpu_summary(model_name) if gpu_client else {}
            gpu_metrics = gpu_metrics or {}  # Handle None return
            
            # Run inference
            resp = vlm_client.vlm.run_image(
                image=image, text=prompt, models=[model_name],
                temperature=config.temperature, max_tokens=config.max_tokens,
            )
            
            if model_name not in resp:
                raise ValueError(f"No response from {model_name}")
            
            model_resp = resp[model_name]
            response_text = model_resp.get('response', '')
            metadata = model_resp.get('metadata', {})
            
            # FIX 2: Use Scorer.compute_all_scores() - it's a classmethod
            scores = Scorer.compute_all_scores(response_text, ground_truth)
            
            # FIX 3: estimate_confidence takes a dict and returns a tuple (score, json)
            conf_score, conf_json = estimate_confidence({'response_text': response_text, 'logprobs': None})
            
            # Cost estimation
            model_spec = next((s for s in model_specs if s.get('name') == model_name), {})
            input_tokens = metadata.get('prompt_tokens', 0) or 0
            output_tokens = metadata.get('completion_tokens', 0) or 0
            cost = (input_tokens * model_spec.get('input_cost_per_1k', 0) + 
                    output_tokens * model_spec.get('output_cost_per_1k', 0)) / 1000
            
            record = {
                'sample_id': sample_id, 'model_name': model_name,
                'model_prefix': model_prefix, 'model_id': model_id,
                'response_raw': response_text, 'response_parsed': response_text,
                'response_length_chars': len(response_text), 'response_length_tokens': output_tokens,
                'input_tokens': input_tokens, 'output_tokens': output_tokens,
                'total_tokens': metadata.get('total_tokens', input_tokens + output_tokens),
                'latency_ms': metadata.get('latency_ms', 0),
                'ok': True, 'error_message': None,
                'stop_reason': metadata.get('finish_reason', 'stop'), 'is_refusal': False,
                # FIX 3: Use unpacked tuple values
                'confidence_score': conf_score,
                'confidence_source': conf_json.get('source'),
                'confidence_reason': conf_json.get('reason'),
                'score_exact_match': scores.get('exact_match', 0.0),
                'score_exact_match_normalized': scores.get('exact_match_normalized', 0.0),
                'score_f1': scores.get('f1', 0.0),
                'score_contains_gt': scores.get('contains_gt', 0.0),
                'score_gt_in_response': scores.get('gt_in_response', 0.0),
                'score_numeric_match': scores.get('numeric_match', 0.0),
                'score_mc_letter_match': scores.get('mc_letter_match', 0.0),
                'is_correct': scores.get('exact_match_normalized', 0.0) > 0.5,
                'pred_answer_letter': None, 'estimated_cost_usd': cost,
                # FIX 1: Correct field names from get_gpu_summary()
                'gpu_name': gpu_metrics.get('gpu_name'), 
                'gpu_index': gpu_metrics.get('gpu_index'),
                'gpu_util_percent': gpu_metrics.get('util_percent'),
                'gpu_mem_used_mb': gpu_metrics.get('mem_used_mb'),
                'gpu_mem_total_mb': gpu_metrics.get('mem_total_mb'),
                'gpu_mem_free_mb': gpu_metrics.get('mem_free_mb'),
                'gpu_temp_celsius': gpu_metrics.get('temp_celsius'),
                'gpu_power_watts': gpu_metrics.get('power_watts'),
                'gpu_power_limit_watts': gpu_metrics.get('power_limit_watts'),
                'gpu_memory_util_percent': gpu_metrics.get('memory_util_percent'),
                'inference_temperature': config.temperature,
                'inference_max_tokens': config.max_tokens, 'inference_top_p': 1.0,
            }
            response_records.append(record)
            
        except Exception as e:
            # Mark as ERROR response
            recovery_logger.error(f"[{model_name}] Error: {e}")
            record = {
                'sample_id': sample_id, 'model_name': model_name,
                'model_prefix': model_prefix, 'model_id': model_id,
                'response_raw': None, 'response_parsed': None,
                'response_length_chars': 0, 'response_length_tokens': 0,
                'input_tokens': 0, 'output_tokens': 0, 'total_tokens': 0, 'latency_ms': 0,
                'ok': False, 'error_message': str(e)[:500],  # Truncate long errors
                'stop_reason': 'error', 'is_refusal': False,
                'confidence_score': None, 'confidence_source': None, 'confidence_reason': None,
                'score_exact_match': 0.0, 'score_exact_match_normalized': 0.0, 'score_f1': 0.0,
                'score_contains_gt': 0.0, 'score_gt_in_response': 0.0, 'score_numeric_match': 0.0,
                'score_mc_letter_match': 0.0, 'is_correct': False, 'pred_answer_letter': None,
                'estimated_cost_usd': 0.0,
                'gpu_name': None, 'gpu_index': None, 'gpu_util_percent': None,
                'gpu_mem_used_mb': None, 'gpu_mem_total_mb': None, 'gpu_mem_free_mb': None,
                'gpu_temp_celsius': None, 'gpu_power_watts': None, 'gpu_power_limit_watts': None,
                'gpu_memory_util_percent': None,
                'inference_temperature': config.temperature,
                'inference_max_tokens': config.max_tokens, 'inference_top_p': 1.0,
            }
            response_records.append(record)
    
    return response_records

print(" Recovery functions defined")


✅ Recovery functions defined


---
#  Section 4: Execute Recovery

In [28]:
# === Cell 8: Main Recovery Loop ===

if config.dry_run:
    print("⏭️ DRY RUN MODE - No changes will be made")
    print(f"   Would recover up to {min(len(issues_df), config.max_recovery_samples)} samples")
    print(f"   - {len(missing_images)} missing images")
    print(f"   - {len(missing_responses)} incomplete responses")
    print(f"   - {len(missing_evals)} missing evaluations")
elif vlm_client is None:
    print(" Cannot run recovery: VLM client not initialized")
else:
    print(f" STARTING RECOVERY for up to {config.max_recovery_samples} samples...")
    print("=" * 70)
    
    # Get samples to process (limit to max)
    samples_to_process = issues_df.head(config.max_recovery_samples)
    
    stats = {'images_recovered': 0, 'responses_recovered': 0, 'errors': 0}
    response_batch = []
    
    for _, row in tqdm(samples_to_process.iterrows(), total=len(samples_to_process), desc="Processing samples"):
        sample_id = row['sample_id']
        source_config = row['source_config']
        source_idx = row['source_index'] or 0
        image_id = row['image_id']
        has_image = row['has_image']
        models_with_responses = row['models_with_responses'] or []
        
        # Step 1: Ensure image exists
        image_bytes = None
        if has_image:
            # Fetch from DB
            with engine.connect() as conn:
                result = conn.execute(text("SELECT image_bytes FROM vlm_images WHERE image_id = :iid"), {'iid': image_id})
                r = result.fetchone()
                if r:
                    image_bytes = r[0]
        
        if image_bytes is None:
            # Fetch from Cauldron
            try:
                idx = int(sample_id.split('_')[1]) if '_' in sample_id else source_idx
            except:
                idx = source_idx
            image_bytes = fetch_image_from_cauldron(source_config, idx, sample_id, image_id)
            if image_bytes:
                stats['images_recovered'] += 1
            else:
                recovery_logger.error(f"Cannot get image for {sample_id}, skipping")
                stats['errors'] += 1
                continue
        
        # Step 2: Run inference for missing models
        models_missing = [m for m in MODEL_NAMES if m not in models_with_responses]
        if models_missing:
            recovery_logger.info(f"[{source_config}] {sample_id}: running inference for {models_missing}")
            
            records = run_inference_for_models(
                sample_id=sample_id,
                prompt=row['prompt_text'],
                ground_truth=row['ground_truth'],
                image_bytes=image_bytes,
                models_to_run=models_missing,
                source_config=source_config,
            )
            
            response_batch.extend(records)
            stats['responses_recovered'] += sum(1 for r in records if r['ok'])
            stats['errors'] += sum(1 for r in records if not r['ok'])
            
            # Batch insert
            if len(response_batch) >= config.batch_size:
                insert_responses(response_batch)
                recovery_logger.info(f"   Batch saved {len(response_batch)} responses")
                response_batch = []
    
    # Final batch
    if response_batch:
        insert_responses(response_batch)
        recovery_logger.info(f"   Final batch saved {len(response_batch)} responses")
    
    print("\n" + "=" * 70)
    print(" RECOVERY COMPLETE:")
    print(f"    Images recovered:    {stats['images_recovered']:>8,}")
    print(f"    Responses recovered: {stats['responses_recovered']:>8,}")
    print(f"    Errors (marked):     {stats['errors']:>8,}")
    print("=" * 70)

🔧 STARTING RECOVERY for up to 10000 samples...


Processing samples:   0%|          | 0/9327 [00:00<?, ?it/s]

16:45:13 | RECOVERY     | INFO    | [aokvqa] aokvqa_931_3807c694: running inference for ['deepseek_ocr']
16:45:13 | RECOVERY     | INFO    | [aokvqa] aokvqa_932_e81e6f16: running inference for ['deepseek_ocr']
16:45:13 | RECOVERY     | INFO    | [aokvqa] aokvqa_933_c1a7a6d8: running inference for ['deepseek_ocr']
16:45:14 | RECOVERY     | INFO    | [aokvqa] aokvqa_934_f0852ef1: running inference for ['deepseek_ocr']
16:45:14 | RECOVERY     | INFO    | [aokvqa] aokvqa_935_47f406fb: running inference for ['deepseek_ocr']
16:45:14 | RECOVERY     | INFO    | [aokvqa] aokvqa_936_b64251da: running inference for ['deepseek_ocr']
16:45:14 | RECOVERY     | INFO    | [aokvqa] aokvqa_937_9e4f3973: running inference for ['deepseek_ocr']
16:45:14 | RECOVERY     | INFO    | [aokvqa] aokvqa_938_8cb49d6f: running inference for ['deepseek_ocr']
16:45:15 | RECOVERY     | INFO    | [aokvqa] aokvqa_939_b3f0f46e: running inference for ['deepseek_ocr']
16:45:15 | RECOVERY     | INFO    | [aokvqa] aokvqa_940


📊 RECOVERY COMPLETE:
   📷 Images recovered:           0
   ✅ Responses recovered:    9,486
   ❌ Errors (marked):            0


In [1]:
# === Cell 9: Run Evaluations for New Responses ===

if config.dry_run:
    print("⏭️ Skipping evaluation recovery (dry run)")
elif eval_pipeline is None:
    print("⏭️ Evaluation pipeline not initialized, skipping")
else:
    print(" Running evaluations for responses missing evaluations...")
    print("=" * 70)
    
    # Get source configs that need evaluation
    configs_needing_eval = missing_evals['source_config'].unique().tolist()  # Limit to 10 configs
    
    for source_config in tqdm(configs_needing_eval, desc="Running evaluations"):
        try:
            eval_pipeline.process_source_config(
                source_config=source_config,
                batch_size=50,
                force=False,
                pbar=tqdm
            )
            recovery_logger.info(f"[{source_config}]  Evaluations complete")
        except Exception as e:
            recovery_logger.error(f"[{source_config}] Evaluation failed: {e}")
    
    print("\n Evaluation recovery complete!")

NameError: name 'config' is not defined

ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/vlm_router_env/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/vlm_router_env/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 341, in dispatch_control
    await self.process_control(msg)
  File "/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/vlm_router_env/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 347, in process_control
    idents, msg = self.session.feed_identities(msg, copy=False)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/vlm_router_

---
#  Section 5: Final Verification

In [ ]:
# === Cell 10: Post-Recovery Verification ===

if not config.dry_run:
    print(" Re-running validation checks...\n")
    
    with engine.connect() as conn:
        issues_after = pd.read_sql(text(query_sample_status), conn, params={'expected_models': config.expected_models})
    
    missing_images_after = issues_after[~issues_after['has_image']]
    missing_responses_after = issues_after[issues_after['response_count'] < config.expected_models]
    missing_evals_after = issues_after[issues_after['eval_count'] < issues_after['response_count']]
    
    print("\n" + "=" * 60)
    print(" BEFORE vs AFTER")
    print("=" * 60)
    print(f"\nMissing Images:     {len(missing_images):>8,}  →  {len(missing_images_after):>8,}")
    print(f"Missing Responses:  {len(missing_responses):>8,}  →  {len(missing_responses_after):>8,}")
    print(f"Missing Evals:      {len(missing_evals):>8,}  →  {len(missing_evals_after):>8,}")
    print("=" * 60)
else:
    print("⏭️ Skipping verification (dry run)")
    print("\n To run actual recovery:")
    print("   1. Set config.dry_run = False in Cell 2")
    print("   2. Re-run all cells")

🔄 Re-running validation checks...


📊 BEFORE vs AFTER

Missing Images:            0  →         0
Missing Responses:     9,327  →     8,827
Missing Evals:             0  →       500


In [ ]:
# === Cell 11: Final Summary ===

print("\n" + "=" * 70)
print(" DATA VALIDATION & RECOVERY COMPLETE")
print("=" * 70)
print(f"\nRun ID: {config.run_id}")
print(f"Dry Run: {config.dry_run}")

print("\nFinal Table Counts:")
with engine.connect() as conn:
    for table in tables:
        result = conn.execute(text(f"SELECT COUNT(*) FROM {table}"))
        count = result.fetchone()[0]
        print(f"   {table:25} {count:>12,} rows")

print("=" * 70)


🏁 DATA VALIDATION & RECOVERY COMPLETE

Run ID: recovery_20251207_163519
Dry Run: False

Final Table Counts:
   vlm_samples                     55,890 rows
   vlm_images                      45,674 rows
   vlm_responses                  279,450 rows
   vlm_evaluations                269,464 rows
